In [1]:
from datasets import load_dataset

dataset = load_dataset("code_x_glue_ct_code_to_text", "python")

dataset


d:\Research Papers\Code_Comment_genartaion\paper2_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['id', 'repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url'],
        num_rows: 251820
    })
    validation: Dataset({
        features: ['id', 'repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url'],
        num_rows: 13914
    })
    test: Dataset({
        features: ['id', 'repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url'],
        num_rows: 14918
    })
})

In [2]:
dataset["train"].features


{'id': Value('int32'),
 'repo': Value('string'),
 'path': Value('string'),
 'func_name': Value('string'),
 'original_string': Value('string'),
 'language': Value('string'),
 'code': Value('string'),
 'code_tokens': List(Value('string')),
 'docstring': Value('string'),
 'docstring_tokens': List(Value('string')),
 'sha': Value('string'),
 'url': Value('string')}

In [3]:
sample = dataset["train"][0]

print("Code:\n")
print(sample["code"])

print("\nDocstring:\n")
print(sample["docstring"])


Code:

def settext(self, text, cls='current'):
        """Set the text for this element.

        Arguments:
            text (str): The text
            cls (str): The class of the text, defaults to ``current`` (leave this unless you know what you are doing). There may be only one text content element of each class associated with the element.
        """
        self.replace(TextContent, value=text, cls=cls)

Docstring:

Set the text for this element.

        Arguments:
            text (str): The text
            cls (str): The class of the text, defaults to ``current`` (leave this unless you know what you are doing). There may be only one text content element of each class associated with the element.


In [4]:
# Remove empty docstrings
dataset = dataset.filter(lambda x: x["docstring"] is not None and len(x["docstring"].strip()) > 0)

dataset


DatasetDict({
    train: Dataset({
        features: ['id', 'repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url'],
        num_rows: 251820
    })
    validation: Dataset({
        features: ['id', 'repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url'],
        num_rows: 13914
    })
    test: Dataset({
        features: ['id', 'repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url'],
        num_rows: 14918
    })
})

In [5]:
dataset = dataset.filter(lambda x: len(x["docstring"].split()) > 5)

dataset


DatasetDict({
    train: Dataset({
        features: ['id', 'repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url'],
        num_rows: 224580
    })
    validation: Dataset({
        features: ['id', 'repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url'],
        num_rows: 12508
    })
    test: Dataset({
        features: ['id', 'repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url'],
        num_rows: 13225
    })
})

In [6]:
train_dataset = dataset["train"].select(range(3000)) 
val_dataset = dataset["validation"].select(range(500)) 
test_dataset = dataset["test"].select(range(500)) 
print(len(train_dataset), len(val_dataset), len(test_dataset))

3000 500 500


In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Salesforce/codet5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Loaded successfully")


d:\Research Papers\Code_Comment_genartaion\paper2_env\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
d:\Research Papers\Code_Comment_genartaion\paper2_env\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
d:\Research Papers\Code_Comment_genartaion\paper2_env\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
d:\Research Papers\Code_Comment_genartaion\paper2_env\Lib\site-packages\transformers\utils\generic.py:309: FutureWarnin

Loaded successfully


In [8]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Using device:", device)


Using device: cuda


In [9]:
MAX_INPUT_LEN = 256
MAX_OUTPUT_LEN = 64

def preprocess(example):
    input_text = "summarize: " + example["code"]
    target_text = example["docstring"]

    model_inputs = tokenizer(
        input_text,
        max_length=MAX_INPUT_LEN,
        padding="max_length",
        truncation=True
    )

    labels = tokenizer(
        target_text,
        max_length=MAX_OUTPUT_LEN,
        padding="max_length",
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [10]:
train_tokenized = train_dataset.map(preprocess)
val_tokenized = val_dataset.map(preprocess)
test_tokenized = test_dataset.map(preprocess)


Map: 100%|██████████| 500/500 [00:00<00:00, 605.67 examples/s]


In [11]:
train_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


In [12]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_loader = DataLoader(
    train_tokenized,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_tokenized,
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=True
)

len(train_loader), len(val_loader)


(375, 63)

In [13]:
from transformers import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)


d:\Research Papers\Code_Comment_genartaion\paper2_env\Lib\site-packages\transformers\optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [16]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
)

trainer.train()


  1%|          | 9/750 [01:04<1:28:56,  7.20s/it]
                                                 
 13%|█▎        | 100/750 [07:55<51:26,  4.75s/it]

{'loss': 0.1138, 'learning_rate': 4.346666666666667e-05, 'epoch': 0.27}


                                                 
 27%|██▋       | 200/750 [15:52<43:18,  4.73s/it]

{'loss': 0.0489, 'learning_rate': 3.68e-05, 'epoch': 0.53}


                                                 
 40%|████      | 300/750 [23:44<35:30,  4.73s/it]

...

                                                 
 93%|█████████▎| 700/750 [55:37<03:55,  4.70s/it]

{'loss': 0.0235, 'learning_rate': 3.5333333333333335e-06, 'epoch': 1.87}


100%|██████████| 750/750 [59:32<00:00,  4.74s/it]






























































                                                 
                                              

100%|██████████| 750/750 [59:52<00:00,  4.74s/it]

                                                 
100%|██████████| 750/750 [59:52<00:00,  4.79s/it]  

{'eval_loss': 0.07286234945058823, 'eval_runtime': 19.6269, 'eval_samples_per_second': 25.475, 'eval_steps_per_second': 3.21, 'epoch': 2.0}
{'train_runtime': 3592.3475, 'train_samples_per_second': 1.67, 'train_steps_per_second': 0.209, 'train_loss': 0.043075462341308594, 'epoch': 2.0}


TrainOutput(global_step=750, training_loss=0.043075462341308594, metrics={'train_runtime': 3592.3475, 'train_samples_per_second': 1.67, 'train_steps_per_second': 0.209, 'train_loss': 0.043075462341308594, 'epoch': 2.0})

In [20]:
from tqdm import tqdm

model.eval()

predictions = []
references = []

BATCH_SIZE = 16

for i in tqdm(range(0, len(test_dataset), BATCH_SIZE)):
    batch = test_dataset[i:i+BATCH_SIZE]

    codes = ["summarize: " + c for c in batch["code"]]
    refs = batch["docstring"]

    inputs = tokenizer(
        codes,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=64,
            num_beams=1   # fast greedy decoding
        )

    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    predictions.extend(preds)
    references.extend(refs)

100%|██████████| 32/32 [02:51<00:00,  5.36s/it]


In [21]:
import sacrebleu

bleu = sacrebleu.corpus_bleu(predictions, [references])
print("BLEU Score:", bleu.score)

BLEU Score: 34.65596034661038


In [22]:
import re

def extract_identifiers(code):
    return set(re.findall(r'\b[A-Za-z_][A-Za-z0-9_]*\b', code))

def detect_entity_hallucination(code, comment):
    code_ids = extract_identifiers(code)
    comment_ids = extract_identifiers(comment)

    hallucinated = comment_ids - code_ids

    # remove common English words
    common_words = {"return", "value", "function", "method", "object"}
    hallucinated = hallucinated - common_words

    return len(hallucinated) > 0

In [23]:
hallucinations = 0

for code, pred in zip(test_dataset["code"], predictions):
    if detect_entity_hallucination(code, pred):
        hallucinations += 1

hallucination_rate = hallucinations / len(predictions)

print("Entity Hallucination Rate:", hallucination_rate)

Entity Hallucination Rate: 0.042


In [24]:
def detect_structural_hallucination(code, comment):
    code_has_loop = any(keyword in code for keyword in ["for ", "while "])
    comment_mentions_loop = any(word in comment.lower() for word in ["loop", "iterate", "iteration"])

    if comment_mentions_loop and not code_has_loop:
        return True

    code_has_if = "if " in code
    comment_mentions_condition = any(word in comment.lower() for word in ["condition", "check", "if"])

    if comment_mentions_condition and not code_has_if:
        return True

    return False

In [25]:
total_hallucinations = 0

for code, pred in zip(test_dataset["code"], predictions):
    entity = detect_entity_hallucination(code, pred)
    structural = detect_structural_hallucination(code, pred)

    if entity or structural:
        total_hallucinations += 1

combined_rate = total_hallucinations / len(predictions)

print("Combined Hallucination Rate:", combined_rate)

Combined Hallucination Rate: 0.102


In [26]:
predictions = []
references = []
confidences = []

for i in range(0, len(test_dataset), BATCH_SIZE):
    batch = test_dataset[i:i+BATCH_SIZE]

    codes = ["summarize: " + c for c in batch["code"]]
    refs = batch["docstring"]

    inputs = tokenizer(
        codes,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=64,
            num_beams=1,
            return_dict_in_generate=True,
            output_scores=True
        )

    preds = tokenizer.batch_decode(outputs.sequences, skip_special_tokens=True)

    # Simple confidence approximation (average logit magnitude)
    scores = outputs.scores
    batch_conf = []

    for step_scores in scores:
        probs = torch.softmax(step_scores, dim=-1)
        max_probs = probs.max(dim=-1).values
        batch_conf.append(max_probs.mean().item())

    predictions.extend(preds)
    references.extend(refs)
    confidences.extend(batch_conf)

In [29]:
filtered_codes = []
filtered_preds = []

for code, pred in zip(test_dataset["code"], predictions):
    entity = detect_entity_hallucination(code, pred)
    structural = detect_structural_hallucination(code, pred)

    if not (entity or structural):
        filtered_codes.append(code)
        filtered_preds.append(pred)

In [30]:
filtered_hallucinations = 0

for code, pred in zip(filtered_codes, filtered_preds):
    if detect_entity_hallucination(code, pred) or detect_structural_hallucination(code, pred):
        filtered_hallucinations += 1

filtered_rate = filtered_hallucinations / len(filtered_preds)

print("Filtered Hallucination Rate:", filtered_rate)
print("Remaining samples after filtering:", len(filtered_preds))

Filtered Hallucination Rate: 0.0
Remaining samples after filtering: 449


In [31]:
mitigated_predictions = []

BATCH_SIZE = 16

model.eval()

for i in range(0, len(test_dataset), BATCH_SIZE):
    batch = test_dataset[i:i+BATCH_SIZE]

    codes = batch["code"]
    prefixed = ["summarize: " + c for c in codes]

    inputs = tokenizer(
        prefixed,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        # First pass (greedy)
        outputs = model.generate(
            **inputs,
            max_length=64,
            num_beams=1
        )

    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # Now check each for hallucination
    for code, pred in zip(codes, preds):

        entity = detect_entity_hallucination(code, pred)
        structural = detect_structural_hallucination(code, pred)

        if entity or structural:
            # Regenerate with beam search
            single_input = tokenizer(
                "summarize: " + code,
                return_tensors="pt",
                truncation=True,
                max_length=256
            ).to(device)

            with torch.no_grad():
                regen_output = model.generate(
                    **single_input,
                    max_length=64,
                    num_beams=4,  # stronger decoding
                    early_stopping=True
                )

            regen_pred = tokenizer.decode(
                regen_output[0],
                skip_special_tokens=True
            )

            mitigated_predictions.append(regen_pred)

        else:
            mitigated_predictions.append(pred)

In [32]:
mitigated_hallucinations = 0

for code, pred in zip(test_dataset["code"], mitigated_predictions):
    if detect_entity_hallucination(code, pred) or detect_structural_hallucination(code, pred):
        mitigated_hallucinations += 1

mitigated_rate = mitigated_hallucinations / len(mitigated_predictions)

print("Mitigated Hallucination Rate:", mitigated_rate)

Mitigated Hallucination Rate: 0.094


In [33]:
import sacrebleu

mitigated_bleu = sacrebleu.corpus_bleu(
    mitigated_predictions,
    [test_dataset["docstring"]]
)

print("Mitigated BLEU:", mitigated_bleu.score)

Mitigated BLEU: 34.52256269255546


In [34]:
def get_allowed_words(code):
    identifiers = extract_identifiers(code)
    return identifiers

In [35]:
strong_mitigated_predictions = []

for code, pred in zip(test_dataset["code"], predictions):

    entity = detect_entity_hallucination(code, pred)
    structural = detect_structural_hallucination(code, pred)

    if entity or structural:

        allowed = get_allowed_words(code)

        single_input = tokenizer(
            "summarize: " + code,
            return_tensors="pt",
            truncation=True,
            max_length=256
        ).to(device)

        with torch.no_grad():
            regen_output = model.generate(
                **single_input,
                max_length=64,
                num_beams=6,
                repetition_penalty=1.2,
                temperature=0.7,
                early_stopping=True
            )

        regen_pred = tokenizer.decode(
            regen_output[0],
            skip_special_tokens=True
        )

        strong_mitigated_predictions.append(regen_pred)

    else:
        strong_mitigated_predictions.append(pred)

d:\Research Papers\Code_Comment_genartaion\paper2_env\Lib\site-packages\transformers\generation\configuration_utils.py:389: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


In [36]:
strong_hallucinations = 0

for code, pred in zip(test_dataset["code"], strong_mitigated_predictions):
    if detect_entity_hallucination(code, pred) or detect_structural_hallucination(code, pred):
        strong_hallucinations += 1

strong_rate = strong_hallucinations / len(strong_mitigated_predictions)

print("Strong Mitigation Hallucination Rate:", strong_rate)

strong_bleu = sacrebleu.corpus_bleu(
    strong_mitigated_predictions,
    [test_dataset["docstring"]]
)

print("Strong Mitigation BLEU:", strong_bleu.score)

Strong Mitigation Hallucination Rate: 0.094
Strong Mitigation BLEU: 34.55896003725563


In [37]:
import numpy as np

print("Average confidence:", np.mean(confidences))
print("Min confidence:", np.min(confidences))
print("Max confidence:", np.max(confidences))

Average confidence: 0.9992823675274849
Min confidence: 0.9221007823944092
Max confidence: 0.9999998211860657


In [38]:
threshold = np.mean(confidences)

In [39]:
confidence_filtered_predictions = []

for code, pred, conf in zip(test_dataset["code"], predictions, confidences):

    entity = detect_entity_hallucination(code, pred)
    structural = detect_structural_hallucination(code, pred)

    if (entity or structural) and conf < threshold:
        # reject low-confidence hallucinations
        continue
    else:
        confidence_filtered_predictions.append(pred)

In [40]:
conf_hallucinations = 0

for code, pred in zip(
    test_dataset["code"][:len(confidence_filtered_predictions)],
    confidence_filtered_predictions
):
    if detect_entity_hallucination(code, pred) or detect_structural_hallucination(code, pred):
        conf_hallucinations += 1

conf_rate = conf_hallucinations / len(confidence_filtered_predictions)

print("Confidence-Filtered Hallucination Rate:", conf_rate)
print("Remaining samples:", len(confidence_filtered_predictions))

Confidence-Filtered Hallucination Rate: 0.7777777777777778
Remaining samples: 495


In [41]:
hallucinated_conf = []
nonhallucinated_conf = []

for code, pred, conf in zip(test_dataset["code"], predictions, confidences):
    hall = detect_entity_hallucination(code, pred) or detect_structural_hallucination(code, pred)
    
    if hall:
        hallucinated_conf.append(conf)
    else:
        nonhallucinated_conf.append(conf)

print("Avg confidence (hallucinated):", np.mean(hallucinated_conf))
print("Avg confidence (non-hallucinated):", np.mean(nonhallucinated_conf))

Avg confidence (hallucinated): 0.9980936997077045
Avg confidence (non-hallucinated): 0.9994691182882058
